[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/channel_cross_sections/whiteadder_cross_sections.ipynb)

# Valley cross sections along the Whiteadder Water, Lammermuir Hills

This notebook draws **cross sections** of the valley of the **Whiteadder Water** at five places downstream of the Whiteadder Reservoir, in the Lammermuir Hills of south-east Scotland, from **1 m lidar**. It shows three things you can reuse elsewhere:

* `lsdtt-fetch-river-lines`: getting a mapped river centreline, here from **OS Open Rivers**, and following it downstream;
* `lsdtt-swath-profile`: a **swath profile** across the valley at each site;
* reading the outputs with **pandas**, plotting them and exporting them.

The workflow is:
1. Install `lsdviztools3` (plotting) and the `lsdtt3` command-line tools.
2. Get the river lines from OS Open Rivers and trace the Whiteadder downstream from the reservoir.
3. Choose five sites along the river (1 km below the reservoir, then every 2 km), and draw a line across the valley at each, perpendicular to the river.
4. Map the region: a 30 m DEM, the catchment above the last site, the river, the sites and a locator map.
5. Download a small **1 m lidar tile** around each site, and take a swath profile along each line.
6. Read, plot and export the swaths with pandas.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

**`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

### Install PyGMT

`lsdviztools3` draws its maps with PyGMT, which needs the GMT library. pip cannot install GMT, so we install PyGMT (and GMT with it) from conda-forge. This is the slowest step: repeated tests suggest it takes around 70 seconds.

In [ ]:
# mamba prints hundreds of lines, so we hide them and only show the end of
# the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3`, the plotting package, is on [PyPI](https://pypi.org/project/lsdviztools3/). It needs Python 3.12 or later, which Colab has.

In [ ]:
!pip install -q "lsdviztools3>=0.4.2"

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

In [ ]:
!lsdtt-swath-profile -v

## Step 1: The river from OS Open Rivers

[OS Open Rivers](https://www.ordnancesurvey.co.uk/products/os-open-rivers) is the Ordnance Survey's free map of the rivers of Great Britain: a centreline for every watercourse, with its name, digitised in the direction of flow. `lsdtt-fetch-river-lines` downloads it once (about 52 MB, kept in a cache folder and reused), cuts out the lines for a box, and prepares them:

* each line runs **downstream**, and has a `from_node` and a `to_node`: the junctions or ends it runs between. Following `to_node` from one line to the next follows the river downstream;
* lines are in the coordinate system we ask for: `output_epsg: 27700`, the **British National Grid** (eastings and northings in metres), which is also what the lidar comes in;
* `feature_class` says what each line is: a `river` or `stream`, or a `lake_connector`, the line Open Rivers draws through a lake or reservoir to keep the network joined.

**A gap below the dam.** Open Rivers breaks the Whiteadder for about 50 m just below the reservoir's dam, so the line leaving the reservoir does not join the line that carries on downstream. `gap_bridge_algorithm_choice: facing_tips` joins two loose line ends that point towards each other across a short gap (up to `max_nonroad_gap_metres`) with a straight `bridge_gap` line, so we can follow the river through.

One box serves the whole notebook: it holds the river lines, and later the 30 m DEM of the catchment.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (55.745, -2.76)
upper_right = (55.955, -2.395)


def write_box(f):
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")


with open("whiteadder_rivers.param", "w") as f:
    f.write("write_fname: whiteadder\n")
    write_box(f)
    f.write("river_line_source: gb_os_open_rivers\n")
    f.write("output_epsg: 27700\n")
    f.write("gap_bridge_algorithm_choice: facing_tips\n")
    f.write("max_nonroad_gap_metres: 60\n")
!lsdtt-fetch-river-lines ./ whiteadder_rivers.param > whiteadder_rivers.log
!grep -E "Feature classes|lake_connector|river |Gap bridging|Network:" whiteadder_rivers.log

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

lines = gpd.read_file("whiteadder_river_lines.fgb")
print(f"{len(lines)} lines; columns: {lines.columns.tolist()}")
print(lines.feature_class.value_counts().to_string())
lines[lines.name == "Whiteadder Water"][["line_id", "name", "feature_class", "from_node", "to_node", "length_m"]].head(10)

### Following the river downstream from the reservoir

We start where the river leaves the reservoir. The reservoir is the `lake_connector` line closest to the dam; the river leaves it at that line's `to_node`. From there we take the line starting at that node, then the line starting at its `to_node`, and so on, for 12 km. Printing the names on the way checks that we are still on the Whiteadder.

In [ ]:
from shapely.geometry import Point
from shapely.ops import linemerge

# The dam at the east end of the Whiteadder Reservoir, as (longitude, latitude) from Google Maps
dam = gpd.GeoSeries([Point(-2.5420, 55.8600)], crs="EPSG:4326").to_crs("EPSG:27700").iloc[0]

connectors = lines[lines.feature_class == "lake_connector"]
reservoir = connectors.loc[connectors.distance(dam).idxmin()]
print(f"Reservoir line {reservoir.line_id}: {reservoir.length_m:.0f} m long, {reservoir.geometry.distance(dam):.0f} m from the dam point")

line_from = {row.from_node: row for row in lines.itertuples()}
node, parts, names = reservoir.to_node, [], []
while node in line_from and sum(p.length for p in parts) < 12000:
    row = line_from[node]
    parts.append(row.geometry)
    names.append(f"{row.name or '(no name)'} [{row.repair_type}]")
    node = row.to_node

river = linemerge(parts)
print(f"Traced {len(parts)} lines, {river.length:.0f} m, as one {river.geom_type}")
print(pd.Series(names).value_counts().to_string())
gpd.GeoDataFrame({"name": ["Whiteadder Water"]}, geometry=[river], crs="EPSG:27700").to_file("whiteadder_river.fgb")

## Step 2: The sites and the cross-section lines

Distance along the river is measured from the reservoir. The sites are at 1, 3, 5, 7 and 9 km.

At each site we need the **direction of the river**: we take the points 50 m upstream and 50 m downstream along the line, and the direction from one to the other. The cross-section line runs at right angles to it, centred on the river, from the **left bank to the right bank, looking downstream**, `section_half_length` metres each way.

The line is the **baseline** of the swath in Step 5: the swath tool measures distances along it from its start, the left-hand end.

In [ ]:
from shapely.geometry import LineString

site_distances_m = [1000, 3000, 5000, 7000, 9000]
section_half_length = 480        # metres either side of the river
direction_window = 50            # metres up- and downstream used for the river's direction

rows = []
for i, d in enumerate(site_distances_m):
    centre = river.interpolate(d)
    up, down = river.interpolate(d - direction_window), river.interpolate(d + direction_window)
    dx, dy = down.x - up.x, down.y - up.y
    dx, dy = dx / np.hypot(dx, dy), dy / np.hypot(dx, dy)          # unit vector, downstream
    left_x, left_y = -dy, dx                                          # 90 degrees anticlockwise: the left bank
    baseline = LineString([(centre.x + section_half_length * left_x, centre.y + section_half_length * left_y),
                           (centre.x - section_half_length * left_x, centre.y - section_half_length * left_y)])
    rows.append({"site": f"S{i + 1}", "distance_from_reservoir_km": d / 1000,
                 "easting": round(centre.x, 1), "northing": round(centre.y, 1),
                 "river_bearing_deg": round(np.degrees(np.arctan2(dx, dy)) % 360, 1),
                 "geometry": baseline})

sections = gpd.GeoDataFrame(rows, crs="EPSG:27700")
sections.to_file("whiteadder_sections.fgb")
sections.drop(columns="geometry").to_csv("whiteadder_sites.csv", index=False)
sections.drop(columns="geometry")

`river_bearing_deg` is the direction the river flows at each site, clockwise from north. `whiteadder_sites.csv` keeps the table; `whiteadder_sections.fgb` keeps the lines, for QGIS or for later steps.

**A note on FlatGeobuf.** A FlatGeobuf file written with a spatial index (the default) stores its features in a **spatial order**, not the order you wrote them. When you read one back, sort it, e.g. `gpd.read_file("whiteadder_sections.fgb").sort_values("site")`.

## Step 3: The regional map

A 30 m Copernicus DEM of the box gives the setting: its hillshade, the **catchment** of the Whiteadder above the last site (from `lsdtt-basin-extraction`, with the last site as the outlet), the traced river, the five lidar tiles of Step 4 (red squares, 1 km across), and a locator map.

In [ ]:
with open("region_fetch.param", "w") as f:
    f.write("write_fname: region\n")
    write_box(f)
    f.write("dem_source: cop30_aws\n")
    f.write("target_epsg: 27700\n")
!lsdtt-fetch-raster ./ region_fetch.param

with open("region_hillshade.param", "w") as f:
    f.write("read_fname: region_DEM.tif\n")
    f.write("write_fname: region_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")
!lsdtt-raster-preprocessing ./ region_hillshade.param

# The outlet is the last site, as latitude and longitude
last = sections.iloc[-1]
outlet = gpd.GeoSeries([Point(last.easting, last.northing)], crs="EPSG:27700").to_crs("EPSG:4326").iloc[0]
pd.DataFrame({"latitude": [outlet.y], "longitude": [outlet.x]}).to_csv("region_outlet.csv", index=False)

with open("region_basin.param", "w") as f:
    f.write("read_fname: region_DEM.tif\n")
    f.write("write_fname: region\n")
    f.write("threshold_contributing_pixels: 500\n")
    f.write("select_basins_from_outlets: true\n")
    f.write("basin_outlet_fname: region_outlet.csv\n")
    f.write("basin_outlet_snapping_method: expanding_square\n")
    f.write("basin_outlet_contributing_pixels_threshold: 20000\n")
    f.write("write_basin_outlines: true\n")
    f.write("write_basin_outlines_as_polygon: true\n")
    f.write("basin_outlines_write_vector_format: flatgeobuf\n")
!lsdtt-basin-extraction ./ region_basin.param > region_basin.log

catchment = gpd.read_file("region_junction_basin_polygons.fgb")
print(f"Catchment above {last.site}: {catchment.area.sum() / 1e6:.1f} km2")

The **lidar tiles** are squares `tile_half_size` metres either side of each site: 500 m, so each tile is 1 km across. The cross-section lines (480 m each way) fit inside them.

The map is put together from `lsdviztools3`'s drawing functions, each adding a layer to the same figure: the hillshade, the catchment, the river, the tiles, and the locator map in a corner.

In [ ]:
from IPython.display import Image, display
from shapely.geometry import box

from lsdviztools3.render.base import save_figure
from lsdviztools3.render.basemap import add_locator_inset
from lsdviztools3.render.basins import render_basins
from lsdviztools3.render.channels import render_channels
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.style import MapStyle

tile_half_size = 500
tiles = gpd.GeoDataFrame({"site": sections.site},
                         geometry=[box(e - tile_half_size, n - tile_half_size, e + tile_half_size, n + tile_half_size)
                                   for e, n in zip(sections.easting, sections.northing)],
                         crs="EPSG:27700")
tiles.to_file("whiteadder_lidar_tiles.fgb")

style = MapStyle(cmap="gray", transparent=False)
fig = render_hillshade("region_hs_hillshade.tif", shade=False, style=style)
render_basins(catchment, fill="lightblue", transparency=80,
              style=MapStyle(line_width="1p", line_color="blue"), fig=fig)
render_channels("whiteadder_river.fgb", style=MapStyle(line_width="1.2p", line_color="blue"), fig=fig, legend=False)
render_basins(tiles, label_col="site", fill="red", transparency=40,
              style=MapStyle(line_width="0.8p", line_color="red"), fig=fig)
add_locator_inset(fig, "region_DEM.tif", mode="regional", position="TL", width="4.5c")
save_figure(fig, "whiteadder_regional_map.png", style=style)
display(Image("whiteadder_regional_map.png", width=800))

## Step 4: 1 m lidar around each site

For each tile, `lsdtt-fetch-raster` downloads the Scottish lidar (`dem_source: gb_sct_lidar`, phase 3, resampled to 1 m) for a box in latitude and longitude. We work out that box from the tile's corners in British National Grid, and ask for the output back in British National Grid (`target_epsg: 27700`). The box in latitude and longitude is slightly larger than the square, so each DEM is a little over 1000 × 1000 pixels. Each tile takes a few seconds.

In [ ]:
for tile in tiles.itertuples():
    west, south, east, north = gpd.GeoSeries([tile.geometry], crs="EPSG:27700").to_crs("EPSG:4326").total_bounds
    with open(f"{tile.site}_fetch.param", "w") as f:
        f.write(f"write_fname: {tile.site}\n")
        f.write(f"west: {west}\neast: {east}\nsouth: {south}\nnorth: {north}\n")
        f.write("dem_source: gb_sct_lidar\n")
        f.write("source_collection_choice: phase-3\n")
        f.write("grid_spacing: 1\n")
        f.write("resampling_method: average\n")
        f.write("target_epsg: 27700\n")
    !lsdtt-fetch-raster ./ {tile.site}_fetch.param > {tile.site}_fetch.log
    !grep -E "Fetched|Source tiles" {tile.site}_fetch.log

## Step 5: A swath profile across the valley at each site

`lsdtt-swath-profile` takes every pixel within `half_width_metres` of a **baseline** line, finds the nearest point on the line, and groups the pixels into bins of `bin_width_metres` by their distance along the line from its start. For each bin it writes the number of pixels, the mean elevation and 21 percentiles (`p0`, `p5`, …, `p100`) to `<write_fname>_swath_profile.csv`.

With our cross-section lines as baselines, each swath is a **cross section of the valley**, 30 m wide (`half_width_metres: 15`), in 2 m bins. A single line of pixels would catch every boulder and bank on that one line; the swath shows the range of elevations across a 30 m strip.

The baseline must be in its own file, in the DEM's coordinate system: we write each site's line to `<site>_baseline.fgb`.

In [ ]:
swath_half_width = 15
swath_bin_width = 2

for section in sections.itertuples():
    sections[sections.site == section.site].to_file(f"{section.site}_baseline.fgb")
    with open(f"{section.site}_swath.param", "w") as f:
        f.write(f"read_fname: {section.site}_DEM.tif\n")
        f.write(f"write_fname: {section.site}\n")
        f.write(f"baseline_vector_fname: {section.site}_baseline.fgb\n")
        f.write(f"half_width_metres: {swath_half_width}\n")
        f.write(f"bin_width_metres: {swath_bin_width}\n")
        f.write("write_distance_rasters: false\n")
    !lsdtt-swath-profile ./ {section.site}_swath.param > {section.site}_swath.log

!ls -l *_swath_profile.csv

## Step 6: Reading the swaths with pandas

Each swath is a CSV file, so `pandas.read_csv` reads it into a table (a `DataFrame`) in one line. Here is the first site's:

In [ ]:
s1 = pd.read_csv("S1_swath_profile.csv")
print(s1.shape, "(rows, columns)")
print(s1.columns.tolist())
s1.head()

`along_axis_centre` is the middle of each bin, in metres from the left-hand end of the line. `n_pixels` is how many pixels went into the bin; `mean` and `p0` … `p100` are the elevations.

**Check the pixel counts.** A 30 m wide swath in 2 m bins should put about 60 pixels in every bin. `describe()` summarises each column:

In [ ]:
s1[["along_axis_centre", "n_pixels", "mean", "p50"]].describe().round(1)

In [ ]:
s1.loc[[0, 1, 2, len(s1) - 3, len(s1) - 2, len(s1) - 1], ["along_axis_centre", "n_pixels", "mean"]]

Are the first and last bins like the others? With **lsdtt3 v0.9.5 and earlier**, they hold several times more pixels: those versions also counted the pixels **beyond the ends** of the line, in a half-circle within `half_width_metres` of each end, in the end bins. Newer versions leave those pixels out. To be safe with either, we leave out any bin with more than one and a half times the typical pixel count; with a newer version this removes nothing.

We also add a column that puts the river at zero: `distance_from_river_m`, negative on the left bank and positive on the right bank, looking downstream.

In [ ]:
def read_swath(site):
    """One site's swath as a DataFrame, without the end bins, with the river at distance 0."""
    profile = pd.read_csv(f"{site}_swath_profile.csv")
    profile = profile[profile.n_pixels <= 1.5 * profile.n_pixels.median()].copy()
    profile.insert(0, "site", site)
    profile.insert(2, "distance_from_river_m", profile.along_axis_centre - section_half_length)
    return profile


swaths = {site: read_swath(site) for site in sections.site}
for site, profile in swaths.items():
    print(f"{site}: {len(profile)} bins, elevation {profile['mean'].min():.1f} to {profile['mean'].max():.1f} m")

### Plotting the swaths

Each panel is one site: the mean elevation in each bin (line), the 5th to 95th percentiles (shaded), and the river (dashed). The vertical scale is exaggerated, and differs between panels.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(swaths), 1, figsize=(10, 3 * len(swaths)), sharex=True)
for ax, (site, profile) in zip(axes, swaths.items()):
    info = sections.set_index("site").loc[site]
    ax.fill_between(profile.distance_from_river_m, profile.p5, profile.p95, color="tab:blue", alpha=0.3,
                    label="5th–95th percentile")
    ax.plot(profile.distance_from_river_m, profile["mean"], color="black", lw=1, label="mean")
    ax.axvline(0, color="tab:blue", ls="--", lw=1)
    ax.set_ylabel("Elevation (m)")
    ax.set_title(f"{site}: {info.distance_from_reservoir_km:.0f} km below the reservoir", loc="left")
axes[0].legend(loc="best")
axes[-1].set_xlabel("Distance from the river (m): left bank < 0 < right bank, looking downstream")
plt.tight_layout()
plt.savefig("whiteadder_swaths.png", dpi=150)
plt.show()

### Where each section is

The lidar hillshade of each tile, with the cross-section line and the swath (red), and the Open Rivers line (blue). We make the hillshades with `lsdtt-raster-preprocessing`.

In [ ]:
import rasterio

for site in sections.site:
    with open(f"{site}_hillshade.param", "w") as f:
        f.write(f"read_fname: {site}_DEM.tif\n")
        f.write(f"write_fname: {site}_hs\n")
        f.write("write_output_format: gtiff\n")
        f.write("write_hillshade_raster: true\n")
    !lsdtt-raster-preprocessing ./ {site}_hillshade.param > {site}_hillshade.log

river_line = gpd.read_file("whiteadder_river.fgb")
fig, axes = plt.subplots(1, len(sections), figsize=(4 * len(sections), 4.5))
for ax, section in zip(axes, sections.itertuples()):
    with rasterio.open(f"{section.site}_hs_hillshade.tif") as src:
        left, bottom, right, top = src.bounds
        ax.imshow(src.read(1, masked=True), cmap="gray", extent=[left, right, bottom, top])   # masked: nodata not drawn
    gpd.GeoSeries([section.geometry.buffer(swath_half_width, cap_style="flat")]).plot(
        ax=ax, facecolor="red", alpha=0.3, edgecolor="red")
    river_line.plot(ax=ax, color="tab:blue", lw=1)
    start = section.geometry.coords[0]
    ax.annotate("left", start, color="red", fontsize=9, ha="center", va="bottom")
    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)
    ax.set_title(section.site)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## Step 7: Exporting the swaths

Each site already has its own file from `lsdtt-swath-profile` (`S1_swath_profile.csv` …). For a spreadsheet, or for another program, one table with all five is often handier. `pd.concat` stacks the five DataFrames (without the end bins, and with the `site` and `distance_from_river_m` columns), and `to_csv` writes it:

In [ ]:
all_swaths = pd.concat(swaths.values(), ignore_index=True)
all_swaths.to_csv("whiteadder_cross_sections.csv", index=False)
print(f"whiteadder_cross_sections.csv: {len(all_swaths)} rows, {all_swaths.site.nunique()} sites")

### Reading the exports back

Anyone with the files can now read them with pandas. Each site's file on its own:

In [ ]:
for csv_file in sorted(glob.glob("S*_swath_profile.csv")):
    df = pd.read_csv(csv_file)
    print(f"{csv_file}: {len(df)} bins, {df['along_axis_centre'].max():.0f} m long, "
          f"lowest mean elevation {df['mean'].min():.1f} m")

And the combined table, split by site with `groupby`:

In [ ]:
combined = pd.read_csv("whiteadder_cross_sections.csv")
combined.groupby("site").agg(n_bins=("mean", "size"),
                             width_m=("distance_from_river_m", lambda x: x.max() - x.min()),
                             lowest_m=("mean", "min"),
                             highest_m=("mean", "max")).round(1)

To take out one site, or one side of the river, filter the rows:

In [ ]:
s3_left_bank = combined[(combined.site == "S3") & (combined.distance_from_river_m < 0)]
s3_left_bank[["distance_from_river_m", "n_pixels", "mean", "p5", "p50", "p95"]].head()

## Questions

* How does the shape of the cross section change from S1 to S5? Compare the width of the valley floor and the height and steepness of the valley sides.
* Where are the two sides of a section different from each other? Look at that section on the tile maps.
* How wide is the shaded band in each panel? Where is it widest, and what does that say about the 30 m strip there?

## Things to try

* **Other sites.** Change `site_distances_m`, for example to every kilometre. The river traced in Step 1 is 12 km long.
* **A wider or narrower swath.** Try `swath_half_width` of 5 or 50 m. How do the mean and the percentile band change?
* **Longer sections.** Make `tile_half_size` and `section_half_length` larger to reach the tops of the valley sides. The tiles then take longer to download.
* **Another river.** Change the box, the `dam` point and the stopping distance in Step 1. Any river in Open Rivers can be traced the same way; start from a point on the river instead of a reservoir by choosing the nearest line of any `feature_class`.
* **Your own lines.** Any line file in British National Grid can be a baseline: draw one in QGIS and save it as FlatGeobuf.

## Every run is recorded

Each program writes `<write_fname>_ingested_params.param`: every parameter it used. You can run it again as it is to repeat the run. Keep these files with your results.

## Data and references

* River lines: OS Open Rivers, © Crown copyright and database right, Ordnance Survey, licensed under the Open Government Licence v3.0; see `whiteadder_ATTRIBUTION.txt`.
* Lidar: Scottish Public Sector LiDAR Phase 3, Scottish Government, from the Scottish Remote Sensing Portal (https://remotesensingdata.gov.scot/). Contains public sector information licensed under the Open Government Licence v3.0.
* DEM: Copernicus GLO-30 (© DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018, provided under COPERNICUS by the European Union and ESA).
* Locator map: drawn with GMT through PyGMT, from GMT's coastline, border and river data (GSHHG). Wessel, P., Luis, J. F., Uieda, L., Scharroo, R., Wobbe, F., Smith, W. H. F. and Tian, D. (2019). The Generic Mapping Tools version 6. *Geochemistry, Geophysics, Geosystems*, 20, 5556–5564. https://doi.org/10.1029/2019GC008515
* Every input of the programs used: https://lsdtopotools.github.io/lsdtt3/ (or run any program with `-h`).